Reading Normalized Data Quickly using a Database
------------------------------------------------------------

In your readonly database, there is an un-normalized table called *home_value_by_zip* with 4,466,776 records of un-normalized data.  There is also a normalized copy of the same data in a few tables in the database.  This goal of this assignment is to use JOIN and GROUP BY to query the normalized data, resulting in faster and more efficient queries.

You must examine the tables using psql and figure out how to connect them and then construct an efficient SQL query in this notebook that will retrieve the requested data.

In [8]:
# https://www.pg4e.com/code/ipynb/03-normalization.ipynb

import psycopg2
import pandas as pd
import time

In [9]:
# Update this SQL connection string with correct readonly database credentials
sql_string = 'dbname=readonly user=readonly password=secret_pg4e host=pg.pg4e.com port=5432'

In [10]:
# open a new connection
# NOTE: this will raise an OperationalError if the readonly database credentials are not updated in the previous cell
conn = psycopg2.connect(sql_string,connect_timeout=3)
print(f"connection created: {conn}")

connection created: <connection object at 0x7ed8efec91c0; dsn: 'user=readonly password=xxx connect_timeout=3 dbname=readonly host=pg.pg4e.com port=5432', closed: 0>


You are to construct a query using the normalized tables that will return the same results as:

    SELECT state, avg(ym_val) AS average FROM home_value_by_zip
    GROUP BY state ORDER BY average DESC LIMIT 10;

This query may run quite long - and might not finish - you should construct a more efficient query using JOIN starting with the `home_value` table.

Here is the expected output of the first few rows of the query:

     state |       average       
    -------+---------------------
     CA    | 429388.882710557533
     HI    | 384304.615036999379
     DC    | 373415.607524148449
     NJ    | 313458.077439427195


In [11]:
### DEFINE YOUR SQL QUERY HERE (NOTE: this query will fail as-is)
state_avg_sql = """
SELECT
    s.state,
    AVG(h.ym_val) AS average
FROM home_value as h
JOIN home_state AS s ON h.state_id = s.id
GROUP BY s.state
ORDER BY average DESC
LIMIT 10;
"""
### END SOLUTION

if state_avg_sql is None:
    raise Exception('Please define the sql query above')

start_time = time.time()
state_avg_df = pd.read_sql_query(state_avg_sql, conn)
print(f"state average execution time: {round(time.time()-start_time, 2)}s")
state_avg_df.head()

/tmp/ipykernel_1011/1756034755.py:18: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  state_avg_df = pd.read_sql_query(state_avg_sql, conn)


state average execution time: 1.78s


,state,average
0,CA,429388.882711
1,HI,384304.615037
2,DC,373415.607524
3,NJ,313458.077439
4,MA,312038.885432


In [12]:
### BEGIN HIDDEN TESTS
assert state_avg_df['state'][1] == 'HI'
assert state_avg_df['average'][1] > 384304
assert state_avg_df['average'][1] < 384305
assert state_avg_df['state'][3] == 'NJ'
assert state_avg_df['average'][3] > 313458
assert state_avg_df['average'][3] < 313459

if state_avg_sql.lower().find('join') < 0 :
    raise Exception('You need to have a JOIN in your query')
### END HIDDEN TESTS

You are to construct a query using the normalized tables that will return the same results as:

    SELECT city, avg(ym_val) AS average FROM home_value_by_zip
    GROUP BY city ORDER BY average DESC LIMIT 10;

This query may run quite long - and might not finish - you should construct a more efficient query using JOIN starting with the `home_value` table.

Here is the expected output of the first few rows of the query:

           city       |       average        
     -----------------+----------------------
      Atherton        | 3625292.526690391459
      Portola Valley  | 2218466.548042704626
      Fisher Island   | 2078791.814946619217
      Montecito       | 1939405.693950177936


In [13]:
### DEFINE YOUR SQL QUERY HERE (NOTE: this query will fail as-is)
city_avg_sql = """
SELECT
    c.city,
    AVG(h.ym_val) AS average
FROM home_value AS h 
JOIN home_city AS c ON h.city_id = c.id
GROUP BY c.city
ORDER BY average DESC
LIMIT 10;
"""
### END SOLUTION

if city_avg_sql is None:
    raise Exception('Please define the sql query above')

start_time = time.time()
city_avg_df = pd.read_sql_query(city_avg_sql, conn)
print(f"state average execution time: {round(time.time()-start_time, 2)}s")
city_avg_df.head()


/tmp/ipykernel_1011/2155584938.py:18: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  city_avg_df = pd.read_sql_query(city_avg_sql, conn)


state average execution time: 1.77s


,city,average
0,Atherton,3.625293e+06
1,Portola Valley,2.218467e+06
2,Fisher Island,2.078792e+06
3,Montecito,1.939406e+06
4,Los Altos,1.919596e+06


In [14]:
### BEGIN HIDDEN TESTS
assert city_avg_df['city'][1] == 'Portola Valley'
assert city_avg_df['average'][1] > 2218466
assert city_avg_df['average'][1] < 2218467
assert city_avg_df['city'][3] == 'Montecito'
assert city_avg_df['average'][3] > 1939405
assert city_avg_df['average'][3] < 1939406

if city_avg_sql.lower().find('join') < 0 :
    raise Exception('You need to have a JOIN in your query')
### END HIDDEN TESTS